# Feature Engineering

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the data
df = pd.read_parquet('../data/nba_cleaned_labeled.parquet')
df.head()

,clock,period,teamId,shotResult,scoreHome,scoreAway,location,description,actionType,actionId,gameId,seconds_remaining,HOME_WINS
0,PT12M00.00S,1,0,NaN,0,0,NaN,Start of 1st Period (7:11 PM EST),period,1,22400001,720,0
1,PT12M00.00S,1,1610612738,NaN,0,0,h,Jump Ball Horford vs. Capela: Tip to Wallace,Jump Ball,2,22400001,720,0
2,PT11M43.00S,1,1610612737,Missed,0,0,v,MISS Risacher 27' 3PT Jump Shot,Missed Shot,3,22400001,703,0
3,PT11M43.00S,1,1610612738,NaN,0,0,h,Tatum BLOCK (1 BLK),NaN,4,22400001,703,0
4,PT11M42.00S,1,1610612737,NaN,0,0,v,Risacher REBOUND (Off:1 Def:0),Rebound,5,22400001,702,0


**CALCULATE total_seconds_remaining for the entire game, not just quarter by quarter**

In [3]:
def get_total_seconds(row):
    period = row['period']
    sec_rem_in_qtr = row['seconds_remaining']
    
    # Regulation logic (Quarters 1-4)
    if period <= 4:
        return (4 - period) * 720 + sec_rem_in_qtr
    else:
        # Overtime logic: Each OT is 5 minutes (300 seconds)
        # We assume time counts down to 0 for the final OT period
        # For simplicity in a WP model, OT is often treated as 'Period 5'
        return sec_rem_in_qtr 

# Apply the logic
df['total_seconds_remaining'] = df.apply(get_total_seconds, axis=1)

# Optimization: convert to int16 to save memory
df['total_seconds_remaining'] = df['total_seconds_remaining'].astype('int16')

# Check the start of each period
period_starts = df.groupby('period')['total_seconds_remaining'].max()
print("\033[1mSeconds at Start of Period:\033[0m")
print(period_starts)

Seconds at Start of Period:
period
1    2880
2    2160
3    1440
4     720
5     300
Name: total_seconds_remaining, dtype: int16


In [4]:
# Calculate the score margin (Home - Away)
df['score_margin'] = df['scoreHome'] - df['scoreAway']

# Get random slice of dataframe to confirm that score_margin column looks good
df.sample(10)

,clock,period,teamId,shotResult,scoreHome,scoreAway,location,description,actionType,actionId,gameId,seconds_remaining,HOME_WINS,total_seconds_remaining,score_margin
71713,PT11M44.00S,1,1610612761,NaN,1,0,h,Barrett Free Throw 1 of 2 (1 PTS),Free Throw,4,22400143,704,1,2864,1
130193,PT09M27.00S,4,1610612750,NaN,82,83,v,SUB: Edwards FOR DiVincenzo,Substitution,400,22400261,567,1,567,-1
159483,PT00M14.40S,3,1610612751,Made,75,68,h,Sharpe 2' Layup (5 PTS),Made Shot,347,22400320,14,1,734,7
120869,PT01M47.00S,3,1610612760,NaN,87,94,h,J. Williams S.FOUL (P3.PN) (M.Lindsay),Foul,367,22400242,107,0,827,-7
86079,PT00M40.40S,4,1610612737,NaN,115,111,h,MISS Risacher Free Throw 2 of 2,Free Throw,457,22400171,40,1,40,4
165507,PT06M27.00S,4,1610612744,Missed,83,78,h,MISS Wiggins 26' 3PT Pullup Jump Shot,Missed Shot,427,22400332,387,1,387,5
56408,PT03M17.00S,1,1610612757,NaN,20,10,v,SUB: Clingan FOR Simons,Substitution,86,22400112,197,1,2357,10
237320,PT08M59.00S,2,1610612765,NaN,42,28,h,Thompson STEAL (5 STL),NaN,142,22400479,539,1,1979,14
12811,PT02M04.00S,2,1610612762,NaN,51,40,v,Markkanen S.FOUL (P1.T4) (P.Fraher),Foul,231,22400026,124,1,1564,11
149465,PT00M30.50S,2,1610612766,Made,49,55,h,Diabaté 1' Cutting Dunk Shot (2 PTS) (Martin 2...,Made Shot,235,22400300,30,0,1470,-6


**POTENTIAL EDA VISUALIZATION**: Now that we have score_margin, we can get the final score_margin from every game and see the distribution of blowouts vs close games.

A simple binary "is_home" (1 for home, 0 for visitor) is the most efficient way to tell the model whether a team is home team or not (for the consideration of home court advantage).

In [5]:
# Convert location 'h'/'v' into a numerical feature
df['is_home'] = (df['location'] == 'h').astype(int)

In [6]:
# Identify if the Lead Changed from the previous row
# We compare the 'sign' of the current margin to the previous margin
df['margin_sign'] = np.sign(df['score_margin'])
df['lead_changed'] = (df['margin_sign'] != df['margin_sign'].shift(1)) & (df['score_margin'] != 0)
df['lead_changed'] = df['lead_changed'].astype(int)

# Clean up the helper column
df.drop(columns=['margin_sign'], inplace=True)

# Get large slice of dataframe to confirm that lead_changed column looks good
# A '1' for every lead change means the feature is working
df.head(50)

,clock,period,teamId,shotResult,scoreHome,scoreAway,location,description,actionType,actionId,gameId,seconds_remaining,HOME_WINS,total_seconds_remaining,score_margin,is_home,lead_changed
0,PT12M00.00S,1,0,NaN,0,0,NaN,Start of 1st Period (7:11 PM EST),period,1,22400001,720,0,2880,0,0,0
1,PT12M00.00S,1,1610612738,NaN,0,0,h,Jump Ball Horford vs. Capela: Tip to Wallace,Jump Ball,2,22400001,720,0,2880,0,1,0
2,PT11M43.00S,1,1610612737,Missed,0,0,v,MISS Risacher 27' 3PT Jump Shot,Missed Shot,3,22400001,703,0,2863,0,0,0
3,PT11M43.00S,1,1610612738,NaN,0,0,h,Tatum BLOCK (1 BLK),NaN,4,22400001,703,0,2863,0,1,0
4,PT11M42.00S,1,1610612737,NaN,0,0,v,Risacher REBOUND (Off:1 Def:0),Rebound,5,22400001,702,0,2862,0,0,0
5,PT11M38.00S,1,1610612737,Missed,0,0,v,MISS Johnson 14' Driving Floating Bank Jump Shot,Missed Shot,6,22400001,698,0,2858,0,0,0
6,PT11M37.00S,1,1610612738,NaN,0,0,h,Horford REBOUND (Off:0 Def:1),Rebound,7,22400001,697,0,2857,0,1,0
7,PT11M24.00S,1,1610612738,Missed,0,0,h,MISS White 28' 3PT Jump Shot,Missed Shot,8,22400001,684,0,2844,0,1,0
8,PT11M22.00S,1,1610612737,NaN,0,0,v,Johnson REBOUND (Off:0 Def:1),Rebound,9,22400001,682,0,2842,0,0,0
9,PT11M17.00S,1,1610612737,NaN,0,0,v,Johnson Traveling Turnover (P1.T1),Turnover,10,22400001,677,0,2837,0,0,0


In [7]:
# Calculate the running total for each specific game
df['total_lead_changes'] = df.groupby('gameId')['lead_changed'].cumsum()

# Get large slice of dataframe to confirm that total_lead_changes column looks good
df.head(50)

,clock,period,teamId,shotResult,scoreHome,scoreAway,location,description,actionType,actionId,gameId,seconds_remaining,HOME_WINS,total_seconds_remaining,score_margin,is_home,lead_changed,total_lead_changes
0,PT12M00.00S,1,0,NaN,0,0,NaN,Start of 1st Period (7:11 PM EST),period,1,22400001,720,0,2880,0,0,0,0
1,PT12M00.00S,1,1610612738,NaN,0,0,h,Jump Ball Horford vs. Capela: Tip to Wallace,Jump Ball,2,22400001,720,0,2880,0,1,0,0
2,PT11M43.00S,1,1610612737,Missed,0,0,v,MISS Risacher 27' 3PT Jump Shot,Missed Shot,3,22400001,703,0,2863,0,0,0,0
3,PT11M43.00S,1,1610612738,NaN,0,0,h,Tatum BLOCK (1 BLK),NaN,4,22400001,703,0,2863,0,1,0,0
4,PT11M42.00S,1,1610612737,NaN,0,0,v,Risacher REBOUND (Off:1 Def:0),Rebound,5,22400001,702,0,2862,0,0,0,0
5,PT11M38.00S,1,1610612737,Missed,0,0,v,MISS Johnson 14' Driving Floating Bank Jump Shot,Missed Shot,6,22400001,698,0,2858,0,0,0,0
6,PT11M37.00S,1,1610612738,NaN,0,0,h,Horford REBOUND (Off:0 Def:1),Rebound,7,22400001,697,0,2857,0,1,0,0
7,PT11M24.00S,1,1610612738,Missed,0,0,h,MISS White 28' 3PT Jump Shot,Missed Shot,8,22400001,684,0,2844,0,1,0,0
8,PT11M22.00S,1,1610612737,NaN,0,0,v,Johnson REBOUND (Off:0 Def:1),Rebound,9,22400001,682,0,2842,0,0,0,0
9,PT11M17.00S,1,1610612737,NaN,0,0,v,Johnson Traveling Turnover (P1.T1),Turnover,10,22400001,677,0,2837,0,0,0,0


POSSESSION INDICATOR

Why this matters for Win Probability:

The "Last Possession" Value: With 10 seconds left in a tie game, the team with the ball has a significantly higher win probability. Without this indicator, the model thinks the game is a 50/50 toss-up.

Efficiency Metrics: This allows you to eventually calculate "Points per Possession," which is a core part of the Matchup Outlook you want for your deep-dive.

**Also include extensive notes about why this was one of the hardest features to engineer ; tons of considerations that require some domain knowledge ; had to take into account possession after events like fouls, timeouts, period starts and ends, etc.**

Explain "Current State" logic and why it is safer for XGBoost model. It is the industry standard because it keeps the "Actor" and the "Possession" aligned one every row

In [8]:
# 'h' or home becomes 1, 'v' or visitor becomes 0. Other rows (like period ends) become NaN
df['possession_indicator'] = np.where(df['location'] == 'h', 1, 
                             np.where(df['location'] == 'v', 0, np.nan))

# Forward-fill to maintain continuity
# This ensures that during timeouts or dead balls, possession stays with the team that had it last
df['possession_indicator'] = df.groupby('gameId')['possession_indicator'].ffill()

Certain fouls should be treated as "non-switching" events.

In [9]:
# CITE SOME CODE ASSISTANCE HERE

# 1. Keep your neutral mask (ensure it catches 'Period End' and 'Period Start')
neutral_actions = ['S.FOUL', 'P.FOUL', 'Timeout', 'Period', 'Jump Ball']
is_neutral = df['description'].str.contains('|'.join(neutral_actions), case=False, na=False)

# 2. Build and implement function to match possession with location in all cases except neutral events
def get_smart_possession(row):
    # IF it's a neutral event (foul/timeout), we don't look at location; we defer to ffill
    if is_neutral.loc[row.name]:
        return np.nan 
    
    # Otherwise, use the acting team's location
    if row['location'] == 'h': return 1
    if row['location'] == 'v': return 0
    
    return np.nan
    
df['possession_indicator'] = df.apply(get_smart_possession, axis=1)

# 3. Handle Period Boundaries with Grouped Filling
# We group by gameId AND period to isolate each quarter's possession state
grouped = df.groupby(['gameId', 'period'])['possession_indicator']

# BACKFILL first: This makes 'Period Start' and 'Jump Ball' rows match 
# whoever performs the first active play of that quarter
df['possession_indicator'] = grouped.bfill()

# FORWARD FILL second: This ensures fouls, timeouts, and 'Period End' rows 
# maintain the state of the play immediately preceding them
df['possession_indicator'] = df.groupby(['gameId', 'period'])['possession_indicator'].ffill()

# 4. Final safety fill and cast
df['possession_indicator'] = df['possession_indicator'].fillna(0).astype('int8')

In [10]:
# Look at an extensive scoring sequence w/ free throws, shots, and fouls to confirm correct implementation
df[['clock', 'location', 'actionType', 'description', 'scoreHome', 'scoreAway', 'possession_indicator']].iloc[80:120]

,clock,location,actionType,description,scoreHome,scoreAway,possession_indicator
80,PT05M36.00S,v,Missed Shot,MISS Mathews 26' 3PT Step Back Jump Shot,12,11,0
81,PT05M35.00S,h,Rebound,Tatum REBOUND (Off:0 Def:4),12,11,1
82,PT05M27.00S,h,Turnover,Holiday Bad Pass Turnover (P1.T3),12,11,1
83,PT05M27.00S,v,NaN,Nance Jr. STEAL (1 STL),12,11,0
84,PT05M17.00S,v,Made Shot,Daniels 6' Driving Layup (2 PTS) (Mathews 1 AST),12,13,0
85,PT04M51.00S,v,Foul,Risacher S.FOUL (P1.T3) (D.Collins),12,13,1
86,PT04M51.00S,h,Free Throw,Brown Free Throw 1 of 2 (9 PTS),13,13,1
87,PT04M51.00S,h,Substitution,SUB: Hauser FOR Holiday,13,13,1
88,PT04M51.00S,h,Free Throw,Brown Free Throw 2 of 2 (10 PTS),14,13,1
89,PT04M39.00S,v,Made Shot,Daniels 8' Driving Floating Jump Shot (4 PTS),14,15,0


In [11]:
# Look at beginning of period to ensure first row is backfilled by first possession

df[['clock', 'location', 'actionType', 'description', 'scoreHome', 'scoreAway', 'possession_indicator']].iloc[:5]

,clock,location,actionType,description,scoreHome,scoreAway,possession_indicator
0,PT12M00.00S,NaN,period,Start of 1st Period (7:11 PM EST),0,0,0
1,PT12M00.00S,h,Jump Ball,Jump Ball Horford vs. Capela: Tip to Wallace,0,0,0
2,PT11M43.00S,v,Missed Shot,MISS Risacher 27' 3PT Jump Shot,0,0,0
3,PT11M43.00S,h,NaN,Tatum BLOCK (1 BLK),0,0,1
4,PT11M42.00S,v,Rebound,Risacher REBOUND (Off:1 Def:0),0,0,0


In [12]:
# Look at start of new quarter to see that possession is

df[['clock', 'location', 'actionType', 'description', 'scoreHome', 'scoreAway', 'possession_indicator']].iloc[134:143]

,clock,location,actionType,description,scoreHome,scoreAway,possession_indicator
134,PT00M00.40S,v,Rebound,Hawks Rebound,31,29,0
135,PT00M00.00S,NaN,period,End of 1st Period (7:38 PM EST),31,29,0
136,PT12M00.00S,NaN,period,Start of 2nd Period (7:41 PM EST),31,29,1
137,PT11M39.00S,h,Made Shot,Pritchard 26' 3PT Jump Shot (3 PTS) (Kornet 1 ...,34,29,1
138,PT11M21.00S,v,Made Shot,Okongwu Dunk (2 PTS) (Johnson 2 AST),34,31,0
139,PT11M06.00S,h,Missed Shot,MISS Pritchard 25' 3PT Jump Shot,34,31,1
140,PT11M05.00S,v,Rebound,Johnson REBOUND (Off:0 Def:3),34,31,0
141,PT10M54.00S,v,Missed Shot,MISS Risacher 25' 3PT Jump Shot,34,31,0
142,PT10M53.00S,h,Rebound,Kornet REBOUND (Off:0 Def:1),34,31,1


In [13]:
# Finally, check rows where an overtime period starts

ot_rows = df.loc[df['period'] == 5]
ot_rows[['clock', 'location', 'actionType', 'description', 'scoreHome', 'scoreAway', 'possession_indicator']].iloc[:5]

,clock,location,actionType,description,scoreHome,scoreAway,possession_indicator
919,PT05M00.00S,NaN,period,Start of 1st OT (9:27 PM EST),111,111,1
920,PT05M00.00S,h,Jump Ball,Jump Ball Stewart vs. Adebayo: Tip to Harris,111,111,1
921,PT04M45.00S,h,Turnover,Beasley Bad Pass Turnover (P4.T15),111,111,1
922,PT04M45.00S,v,NaN,Herro STEAL (2 STL),111,111,0
923,PT04M29.00S,v,Missed Shot,MISS Highsmith 24' 3PT Jump Shot,111,111,0


In NBA analytics, "Clutch" is typically defined as the last 5 minutes of a game when the score is within 5 points. This helps the model prioritize the importance of a single possession or turnover.

In [14]:
# Clutch: Score margin <= 5 AND less than 300 seconds (5 mins) remaining
df['is_clutch'] = ((df['score_margin'].abs() <= 5) & 
                   (df['total_seconds_remaining'] <= 300)).astype(int)

# Locate game sequence that qualifies as clutch
clutch_rows = df.loc[df['is_clutch'] == 1]
clutch_rows[:5] # Clutch sequence starts around row 418

# Check sequence surrounding row 418 before to confirm is_clutch flag triggered at right time
df[['clock', 'period', 'total_seconds_remaining', 'scoreHome', 'scoreAway', 'score_margin', 'is_clutch']].iloc[410:430]

,clock,period,total_seconds_remaining,scoreHome,scoreAway,score_margin,is_clutch
410,PT06M29.00S,4,389,99,99,0,0
411,PT06M08.00S,4,368,101,99,2,0
412,PT05M56.00S,4,356,101,99,2,0
413,PT05M55.00S,4,355,101,99,2,0
414,PT05M43.00S,4,343,101,99,2,0
415,PT05M43.00S,4,343,101,99,2,0
416,PT05M25.00S,4,325,101,101,0,0
417,PT05M03.00S,4,303,104,101,3,0
418,PT04M51.00S,4,291,104,104,0,1
419,PT04M38.00S,4,278,104,104,0,1


"Garbage Time" occurs when the lead is so large that the outcome is statistically certain. In these moments, teams often play bench players and stop using strategic fouls, which can create "noise" in your training data.

A common quantitative threshold for Garbage Time is:

4th Quarter (or OT)

Lead > 20 points with 6 minutes (360 seconds) left

Lead > 10 points with 2 minutes (120 seconds) left

In [15]:
def check_garbage_time(row):
    margin = abs(row['score_margin'])
    sec_rem = row['total_seconds_remaining']
    
    # Only consider 4th quarter or later
    if row['period'] >= 4:
        if sec_rem <= 120 and margin > 10: return 1 # 2 mins left, 10+ point lead
        if sec_rem <= 360 and margin > 20: return 1 # 6 mins left, 20+ point lead
        if margin > 30: return 1                   # Any time in 4th with 30+ lead
    return 0

df['garbage_time_flag'] = df.apply(check_garbage_time, axis=1)

# Locate game sequence that qualifies as garbage time
garbage_time_rows = df.loc[df['garbage_time_flag'] == 1]
garbage_time_rows[:5] # Clutch sequence starts around row 1423

# Check sequence surrounding row 1423 before to confirm garbage_time_flag triggered at right time
df[['clock', 'period', 'total_seconds_remaining', 'scoreHome', 'scoreAway', 'score_margin', 'garbage_time_flag']].iloc[1415:1435]

,clock,period,total_seconds_remaining,scoreHome,scoreAway,score_margin,garbage_time_flag
1415,PT06M50.00S,4,410,100,70,30,0
1416,PT06M29.00S,4,389,100,72,28,0
1417,PT06M14.00S,4,374,100,72,28,0
1418,PT06M14.00S,4,374,100,72,28,0
1419,PT06M14.00S,4,374,100,72,28,0
1420,PT06M14.00S,4,374,100,72,28,0
1421,PT06M04.00S,4,364,100,72,28,0
1422,PT06M02.00S,4,362,100,72,28,0
1423,PT05M55.00S,4,355,100,74,26,1
1424,PT05M34.00S,4,334,102,74,28,1


NOTES about process of choosing baseline context for the model ; how ELO rating wasn't feasible due to the labor-intensiveness and the "cold start" issue for initializing ELOs at the beginning of seasons ; how avg_margin_diff was settled on due to granularity (recognizes point differentials), predictive power (similar to Net Rating which NBA analytics considers a stable and accurate predictor of future wins), and the "lucky" team filter (helps model recognize when teams have a high win percentage but a low point differential meaning they are getting lucky in close games).

In [16]:
# CITE CODE ASSISTANCE HERE

# 1. Identify Home/Away IDs (same as before)
team_mapping = df.groupby('gameId').apply(lambda x: pd.Series({
    'teamIdHome': x.loc[x['location'] == 'h', 'teamId'].iloc[0],
    'teamIdAway': x.loc[x['location'] == 'v', 'teamId'].iloc[0]
}), include_groups=False).reset_index()

# 2. Get unique game results - sorting by gameId instead of date
game_results = df.groupby('gameId').agg({
    'score_margin': 'last'
}).reset_index().sort_values('gameId') # Essential for chronological logic

# Merge mapping
game_results = game_results.merge(team_mapping, on='gameId', how='left')

# 3. Apply the 20-point cap
game_results['capped_margin'] = np.clip(game_results['score_margin'], -20, 20)

# 4. Long Format transformation
home_stats = game_results[['gameId', 'teamIdHome', 'capped_margin']].copy()
home_stats.columns = ['gameId', 'teamId', 'margin']

away_stats = game_results[['gameId', 'teamIdAway', 'capped_margin']].copy()
away_stats.columns = ['gameId', 'teamId', 'margin']
away_stats['margin'] = -away_stats['margin'] 

# 5. Calculate Expanding Mean and Games Played
# We sort by teamId and gameId to ensure the count and mean follow the season timeline
team_history = pd.concat([home_stats, away_stats]).sort_values(['teamId', 'gameId'])

# Calculate the mean of PRIOR games
team_history['avg_margin'] = (
    team_history.groupby('teamId')['margin']
    .apply(lambda x: x.expanding().mean().shift(1))
    .reset_index(level=0, drop=True)
)

# NEW: Calculate games played PRIOR to the current game
# cumcount() returns 0 for the 1st game, 1 for the 2nd, etc.
team_history['games_played'] = team_history.groupby('teamId').cumcount()

# 6. Merge both Strength and Games Played into game_results
# We update the merge to include 'games_played' for both teams
game_results = game_results.merge(
    team_history[['gameId', 'teamId', 'avg_margin', 'games_played']], 
    left_on=['gameId', 'teamIdHome'], right_on=['gameId', 'teamId'], how='left'
).rename(columns={'avg_margin': 'home_strength', 'games_played': 'home_games_played'}).drop(columns='teamId')

game_results = game_results.merge(
    team_history[['gameId', 'teamId', 'avg_margin', 'games_played']], 
    left_on=['gameId', 'teamIdAway'], right_on=['gameId', 'teamId'], how='left'
).rename(columns={'avg_margin': 'away_strength', 'games_played': 'away_games_played'}).drop(columns='teamId')

# Calculate the final differential
game_results['avg_margin_diff'] = game_results['home_strength'] - game_results['away_strength']

# 7. Update your lookup to include the new game count features
final_lookup = game_results[[
    'gameId', 'avg_margin_diff', 'home_games_played', 'away_games_played'
]].drop_duplicates(subset=['gameId'])

# Drop old versions if they exist to avoid duplicate column errors
cols_to_drop = ['avg_margin_diff', 'home_games_played', 'away_games_played']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Perform the final merge into the play-by-play DataFrame
df = df.merge(final_lookup, on='gameId', how='left')

# Fill NaNs with 0 (essential for Game 1 of the season)
df[['avg_margin_diff', 'home_games_played', 'away_games_played']] = \
    df[['avg_margin_diff', 'home_games_played', 'away_games_played']].fillna(0)

# 8. Final verification
print(f"Inconsistent games: {(df.groupby('gameId')['avg_margin_diff'].nunique() > 1).sum()}")

Inconsistent games: 0


In [17]:
# Check a single team (Boston Celtics) to confirm correct implementation of avg_margin_diff
team_check = team_history[team_history['teamId'] == 1610612738].sort_values('gameId').head(5)
print(team_check[['gameId', 'margin', 'avg_margin']])

      gameId  margin  avg_margin
0   22400001      -1         NaN
20  22400021       3   -1.000000
27  22400028      12    1.000000
46  22400047       9    4.666667
60  22400061      20    5.750000


The results confirm that the logic for avg_margin_diff is working exactly as intended.

Game 22400001: The avg_margin is NaN. This is correct because this is the team's first game of the season; they have no prior history to calculate an average from.

Game 22400021: The avg_margin is -1.000000. This is the exact margin from their first game. This proves .shift(1) is correctly using only past data.

Game 22400028: The avg_margin is 1.000000. This is the average of the first two games: $(-1 + 3) / 2 = 1$.

Game 22400047: The avg_margin is 4.666667. This is the average of the first three games: $(-1 + 3 + 12) / 3 = 14 / 3 = 4.666...$

Game 22400061: The avg_margin is 5.750000. This is the average of the first four games: $(-1 + 3 + 12 + 9) / 4 = 23 / 4 = 5.75$.

In [18]:
list(df.columns)

['clock',
 'period',
 'teamId',
 'shotResult',
 'scoreHome',
 'scoreAway',
 'location',
 'description',
 'actionType',
 'actionId',
 'gameId',
 'seconds_remaining',
 'HOME_WINS',
 'total_seconds_remaining',
 'score_margin',
 'is_home',
 'lead_changed',
 'total_lead_changes',
 'possession_indicator',
 'is_clutch',
 'garbage_time_flag',
 'avg_margin_diff',
 'home_games_played',
 'away_games_played']

This next step is the culmination of Notebooks 01 and 02: building a consolidated "Engine" that will ingest the full datasets (~2500 games worth of play-by-play data for seasons 2023-2024 and 2024-2025), clean and parse them, and implement the feature engineering to produce a "model-ready" Parquet file.

In [19]:
import re

def parse_nba_clock(clock_str):
    if not isinstance(clock_str, str): return 0
    match = re.search(r'PT(\d+)M(\d+)', clock_str)
    if match:
        return (int(match.group(1)) * 60) + int(match.group(2))
    return 0

def get_total_seconds(row):
    period = row['period']
    sec_rem = row['seconds_remaining']
    return (4 - period) * 720 + sec_rem if period <= 4 else sec_rem

def check_garbage_time(row):
    margin, sec, period = abs(row['score_margin']), row['total_seconds_remaining'], row['period']
    if period >= 4:
        if (sec <= 120 and margin > 10) or (sec <= 360 and margin > 20) or (margin > 30):
            return 1
    return 0

def nba_engine_processor(df):
    # 1. CLEANING & FORMATTING
    dropped_columns = ['teamTricode', 'personId', 'playerName', 'playerNameI', 
                       'xLegacy', 'yLegacy', 'shotDistance', 'isFieldGoal', 
                       'pointsTotal', 'subType', 'videoAvailable', 'shotValue', 'actionNumber']
    
    df = df.drop(columns=[c for c in dropped_columns if c in df.columns])
    df[['scoreHome', 'scoreAway']] = df[['scoreHome', 'scoreAway']].replace('', np.nan)
    df = df.sort_values(['gameId', 'actionId'])
    
    # Fill scores forward within game
    df['scoreHome'] = df.groupby('gameId')['scoreHome'].ffill().fillna(0)
    df['scoreAway'] = df.groupby('gameId')['scoreAway'].ffill().fillna(0)
    
    # Type Casting
    dtype_mapping = {'actionId': 'int32', 'period': 'int8', 'scoreHome': 'int16', 
                     'scoreAway': 'int16', 'location': 'category', 'actionType': 'category', 
                     'shotResult': 'category'}
    df = df.astype({k: v for k, v in dtype_mapping.items() if k in df.columns})
    
    # 2. CORE TIME & SCORE FEATURES
    df['seconds_remaining'] = df['clock'].apply(parse_nba_clock).astype('int16')
    df['total_seconds_remaining'] = df.apply(get_total_seconds, axis=1).astype('int16')
    df['score_margin'] = df['scoreHome'] - df['scoreAway']
    df['is_home'] = (df['location'] == 'h').astype(int)
    
    # Lead Changes
    df['margin_sign'] = np.sign(df['score_margin'])
    df['lead_changed'] = (df['margin_sign'] != df['margin_sign'].shift(1)) & (df['score_margin'] != 0)
    df['lead_changed'] = (df['lead_changed'] & (df['gameId'] == df['gameId'].shift(1))).astype(int)
    df['total_lead_changes'] = df.groupby('gameId')['lead_changed'].cumsum()
    
    # 3. SMART POSSESSION
    neutral_actions = ['S.FOUL', 'P.FOUL', 'Timeout', 'Period', 'Jump Ball']
    is_neutral = df['description'].str.contains('|'.join(neutral_actions), case=False, na=False)
    
    df['possession_indicator'] = np.nan
    df.loc[~is_neutral & (df['location'] == 'h'), 'possession_indicator'] = 1
    df.loc[~is_neutral & (df['location'] == 'v'), 'possession_indicator'] = 0
    
    # Backfill then Forward fill within Period
    df['possession_indicator'] = df.groupby(['gameId', 'period'])['possession_indicator'].bfill()
    df['possession_indicator'] = df.groupby(['gameId', 'period'])['possession_indicator'].ffill().fillna(0).astype('int8')
    
    # 4. LEVERAGE FLAGS
    df['is_clutch'] = ((df['score_margin'].abs() <= 5) & (df['total_seconds_remaining'] <= 300)).astype(int)
    df['garbage_time_flag'] = df.apply(check_garbage_time, axis=1).astype('int8')
    
    # 5. TARGET LABEL (HOME_WINS)
    win_map = df.groupby('gameId')['score_margin'].last().gt(0).astype(int).to_dict()
    df['HOME_WINS'] = df['gameId'].map(win_map)
    
    return df.drop(columns=['margin_sign'])

This final block processes seasons separately before concatenating.

The baseline context feature avg_margin_diff is calculated in this block for every team for both seasons. Reset games_played and avg_margin to 0/NaN at the start of the 2024 season. This reflects how rosters can change drastically between seasons (trades, draft, free agency).

In [20]:
# 1. Load your separate CSVs
df_23 = pd.read_csv('../data/nbastatsv3_2023.csv')
df_24 = pd.read_csv('../data/nbastatsv3_2024.csv')

# 2. Process them separately to ensure Strength/Games Played reset for the new season
processed_23 = nba_engine_processor(df_23)
processed_24 = nba_engine_processor(df_24)

# CITE CODE ASSISTANCE HERE
# 3. Calculate Strength for each (Resetting for 2024)
def calculate_strengths(season_df):
    mapping = season_df.groupby('gameId').apply(lambda x: pd.Series({
        'teamIdHome': x.loc[x['location'] == 'h', 'teamId'].iloc[0],
        'teamIdAway': x.loc[x['location'] == 'v', 'teamId'].iloc[0]
    }), include_groups=False).reset_index()
    
    results = season_df.groupby('gameId').agg({'score_margin': 'last'}).reset_index()
    results = results.merge(mapping, on='gameId').sort_values('gameId')
    results['capped_margin'] = np.clip(results['score_margin'], -20, 20)
    
    # Long Format for ELO-style calculation
    h = results[['gameId', 'teamIdHome', 'capped_margin']].rename(columns={'teamIdHome': 'teamId', 'capped_margin': 'margin'})
    v = results[['gameId', 'teamIdAway', 'capped_margin']].rename(columns={'teamIdAway': 'teamId', 'capped_margin': 'margin'})
    v['margin'] = -v['margin']
    
    hist = pd.concat([h, v]).sort_values(['teamId', 'gameId'])
    hist['avg_margin'] = hist.groupby('teamId')['margin'].apply(lambda x: x.expanding().mean().shift(1)).reset_index(level=0, drop=True)
    hist['games_played'] = hist.groupby('teamId').cumcount()
    
    # --- ADD THESE LINES TO FIX THE KEYERROR ---
    # Merge home stats from hist
    results = results.merge(
        hist[['gameId', 'teamId', 'avg_margin', 'games_played']], 
        left_on=['gameId', 'teamIdHome'], right_on=['gameId', 'teamId'], how='left'
    ).rename(columns={'avg_margin': 'h_s', 'games_played': 'home_games_played'}).drop(columns='teamId')

    # Merge away stats from hist
    results = results.merge(
        hist[['gameId', 'teamId', 'avg_margin', 'games_played']], 
        left_on=['gameId', 'teamIdAway'], right_on=['gameId', 'teamId'], how='left'
    ).rename(columns={'avg_margin': 'a_s', 'games_played': 'away_games_played'}).drop(columns='teamId')

    # Calculate the differential
    results['avg_margin_diff'] = results['h_s'] - results['a_s']
    # -------------------------------------------
    
    # Now this merge will work because the columns exist in 'results'!
    merged_df = season_df.merge(
        results[['gameId', 'avg_margin_diff', 'home_games_played', 'away_games_played']], 
        on='gameId', 
        how='left'
    )
    
    cols_to_fill = ['avg_margin_diff', 'home_games_played', 'away_games_played']
    merged_df[cols_to_fill] = merged_df[cols_to_fill].fillna(0)
    
    return merged_df

# Apply strength logic
final_23 = calculate_strengths(processed_23)
final_24 = calculate_strengths(processed_24)

# 4. FINAL CONCATENATION & DATA HYGIENE
df_model = pd.concat([final_23, final_24], ignore_index=True)

# Drop non-numeric columns for XGBoost (keeping gameId for splitting)
cols_to_drop = ['clock', 'description', 'location', 'actionType', 'shotResult', 'teamId', 'actionId', 'seconds_remaining']
df_model = df_model.drop(columns=[c for c in cols_to_drop if c in df_model.columns])

# Create a dictionary of optimized types
optimized_dtypes = {
    'is_home': 'int8',
    'lead_changed': 'int8',
    'total_lead_changes': 'int16',
    'is_clutch': 'int8',
    'HOME_WINS': 'int8',
    'home_games_played': 'int16',
    'away_games_played': 'int16',
    'avg_margin_diff': 'float32',
    'gameId': 'object' # Kept as object for ID purposes
}

# Apply the conversion
df_model = df_model.astype(optimized_dtypes)

# Final Save to Parquet
df_model.to_parquet('../data/nba_wp_model_ready_2500.parquet')
print("Engine Complete. File saved for Notebook 03.")

Engine Complete. File saved for Notebook 03.


Perform a "Macro-Micro" check of the resulting Parquet file: look at the high-level distributions to ensure no data was lost and also "spot-check" specific transition points, including the season opener.

In [21]:
# Load the file
df_verify = pd.read_parquet('../data/nba_wp_model_ready_2500.parquet')

# Find the first game of the 2024 season (the transition point)
# Assuming 2024 gameIds start with '00224' or simply appear after the 2023 IDs
first_game_24 = df_24['gameId'].unique()[0]

print(f"Checking Season Transition at Game: {first_game_24}")
check_24 = df_verify[df_verify['gameId'] == first_game_24].head(1)
print(check_24[['avg_margin_diff', 'home_games_played', 'away_games_played']])

Checking Season Transition at Game: 22400001
        avg_margin_diff  home_games_played  away_games_played
598705              0.0                  0                  0


In [22]:
# Check the ranges of your key features
cols_to_check = ['score_margin', 'avg_margin_diff', 'total_seconds_remaining']
print(df_verify[cols_to_check].describe())

       score_margin  avg_margin_diff  total_seconds_remaining
count  1.208117e+06     1.208117e+06             1.208117e+06
mean   1.102305e+00    -2.519331e-02             1.399064e+03
std    1.189128e+01     7.307838e+00             8.347605e+02
min   -5.900000e+01    -2.510000e+01             0.000000e+00
25%   -6.000000e+00    -4.804147e+00             6.820000e+02
50%    1.000000e+00     0.000000e+00             1.411000e+03
75%    8.000000e+00     4.820228e+00             2.125000e+03
max    6.400000e+01     3.200000e+01             2.880000e+03


This looks statistically sound. They confirm that the data hasn't been corrupted by outliers or "hallucinated" values during the feature engineering process.

**score_margin**: A min of -59 and a max of +64 is spot-on for the NBA. These represent rare, massive blowouts. The mean of 1.10 reflects the historical slight "Home Court Advantage" (since the margin is Home - Away).

**avg_margin_diff**: A min of -25 and a max of +32 is excellent. Even individual game results were capped at 20, the difference between a very good team (+15 avg) and a very bad team (-15 avg) can naturally reach around 30.

**total_seconds_remaining**: 
* Max of 2880: This is exactly $48 \text{ minutes} \times 60 \text{ seconds}$. 
* Min of 0: This shows clock parsing handled the final buzzer correctly.
* Mean of 1399: This is roughly the middle of the 2nd quarter, which is exactly where the mathematical average of a basketball game should sit.

**Median (50%) of 0.00 for avg_margin_diff**: This indicates that a significant portion of the data (especially at the start of both the 2023 and 2024 seasons) is correctly initialized at 0 before the teams establish their season averages.

In [23]:
df_verify.dtypes

period                        int8
scoreHome                    int16
scoreAway                    int16
gameId                       int64
total_seconds_remaining      int16
score_margin                 int16
is_home                       int8
lead_changed                  int8
total_lead_changes           int16
possession_indicator          int8
is_clutch                     int8
garbage_time_flag             int8
HOME_WINS                     int8
avg_margin_diff            float32
home_games_played            int16
away_games_played            int16
dtype: object

No "Object (string)" columns are hiding in the training features, which means the data is ready to be ingested by an XGBoost model.

**FINAL NOTES ABOUT WHAT WILL BE DONE IN THE NEXT NOTEBOOKS**